# $\zeta(5)$: historical constructions and new result

Samples are taken every 10 depths, through depth 1000 for Zudilin (2002), 300 for Brown–Zudilin, and 100 for the local trajectory. Each computational example
prints the delta at its final sampled depth; the figure at the end plots
$\delta_N$ against $N$, where
$$D_N=-\log_{10}\left|\zeta(5)-\frac{p_N}{q_N}\right|,
\qquad \delta_N=\frac{D_N}{\log_{10}q_N}-1.$$
Here $p_N/q_N$ is reduced. Accuracy digits measure absolute approximation error.

Run the cells in order (Python 3.11+). The CMF encodings come from the supplied
exploratory notebooks and local supplementary material. Published bounds and the
finite-depth $\delta_N$ values are different quantities; the computations below do
not by themselves prove a published irrationality-measure bound.

In [1]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 matplotlib==3.11.2

## Setup

Each example walks the full 15-axis, rank-7 ${}_8F_7(1)$ CMF exactly.
Zudilin (2002) uses two direct hypergeometric forms and a determinant
readout; the other constructions use the dual walk.


In [2]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx
from flint import fmpq
from cmfs import (zudilin_zeta5_hypergeometric_seeds,
                  zudilin_zeta5_hypergeometric_readout)
from ramanujantools import Position
from ramanujantools.cmf.pfq import pFq
from ramanujantools.flint_core.numeric_matrix import NumericMatrix

sys.set_int_max_str_digits(0)
with ctx.workdps(81_000):  # q_N reaches about 18,000 digits at depth 300 (Brown-Zudilin)
    ZETA5 = arb(5).zeta()
DEPTHS = list(range(10, 3001, 10))
n = sp.Symbol("n")
x = sp.symbols("x0:8")
y = sp.symbols("y0:7")

# Brown-Zudilin's projection: p_N = NUMERATOR . s and q_N = DENOMINATOR . s.
NUMERATOR = [fmpq(16342893984375, 8), fmpq(7888732953125, 2), fmpq(24573697734375, 8),
             1206481062500, fmpq(1912606640625, 8), fmpq(76504265625, 4), 0]
DENOMINATOR = [1969765000000, fmpq(15220831421875, 4), fmpq(11858161171875, 4),
               fmpq(2329422984375, 2), fmpq(461663671875, 2), 18466546875, 0]
results = {}


def delta(approximant):
    """delta with |ζ(5) - p/q| = q^-(1 + delta), for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):  # enough bits to resolve |ζ(5) - p/q|
        error = abs(ZETA5 - arb(p) / q)
        assert error > 0
        return float((-1 - error.log() / arb(q).log()).mid())

## 1. Zudilin (2002): full hypergeometric CMF

W. Zudilin, *A third-order Apéry-like recursion for $\zeta(5)$*,
[equations (7), (15), and Theorem 1](https://arxiv.org/abs/math/0206178).
The published result is exponential convergence, not an irrationality-measure bound:
$$\lim_{N\to\infty}|\zeta(5)-p_N/q_N|^{1/N}=|\mu_2/\mu_3|<1.42521964\cdot10^{-4},$$
where $\mu_2\approx0.33753726$, $\mu_3\approx-2368.31752213$ are roots of
$\mu^3+2368\mu^2-752\mu-16=0$.

**Full CMF:** ${}_8F_7(1)$, rank 7, with all 15 independent axes
$(x_0,\ldots,x_7;y_0,\ldots,y_6)$.
Set
$$F_N(z)={}_8F_7\!\left(\begin{matrix}3N+2,N+1,N+1,N+1,N+1,N+1,N+1,N+1\\
2N+2,2N+2,2N+2,2N+2,2N+2,2N+2,2N+2\end{matrix};z\right),
\quad C_N=\frac{N!^{11}(3N+1)!}{(2N+1)!^7},\quad\theta=z\frac{d}{dz}.$$
Shifting the summation variable in (7) by $k=m+N+1$ gives exactly
$$r_N=C_N[(\theta+3N/2+1)F_N]_{z=1},$$
$$\widetilde r_N=-C_N[(\theta+N+1)(\theta+2N+1)(\theta+3N/2+1)F_N]_{z=1}.$$

The state is $(F,\theta F,\ldots,\theta^6F)$ at $z=1$.
If $T(x,y)$ is the companion matrix of
$\theta\prod_j(\theta+y_j-1)-\prod_i(\theta+x_i)$, the axis matrices supplied by
`pFq(8, 7, 1)` are
$$M_{x_i}=I+T/x_i,\qquad
M_{y_j}=\left(I+T(x,y+e_j)/y_j\right)^{-1}.$$
These are independent contiguous parameter shifts; their conservation follows from
the commuting shift operators. The historical direction is one path in this full field.

**Start:** $(8,3,3,3,3,3,3,3;6,6,6,6,6,6,6)$, corresponding to source index 2.
**Direction:** $(3,1,1,1,1,1,1,1;2,2,2,2,2,2,2)$.
Thus source index $N$ means **$N-2$ walk steps**. At the start the parametric excess
is 13, so all seven derivatives converge. Index 0 has divergent higher derivatives;
indices 0 and 1 remain covered by the independent recurrence check.

The exact seed coefficient rows are obtained in `cmfs.py` by partial fractions of
$$\frac{(5!)^7}{7!(2!)^7}\frac{(m+1)(m+2)\cdots(m+7)}{(m+3)^7(m+4)^7(m+5)^7}.$$
Multiplying by $m^j$ and summing gives $\theta^jF_2(1)$.
After walking, write $r_N/C_N=A_0+A_3\zeta(3)+A_5\zeta(5)$ and
$\widetilde r_N/C_N=B_0+B_3\zeta(3)+B_5\zeta(5)$.
All other zeta coefficient rows cancel exactly. Equation (15) reads
$$\frac{p_N}{q_N}=\frac{A_3B_0-B_3A_0}{A_5B_3-B_5A_3}.$$
The common factor $C_N^2$ cancels.

`local_checks/verify_zudilin_zeta5_full_cmf.py` checks every axis pair at two off-trajectory
rational points, source normalization at indices 2 through 9, and exact agreement
with the published scalar recurrence at every sampled index through 1000.
Two Arb precisions verify $\delta_{1000}=-0.218905438894011\ldots$;
the reduced denominator has 4923 digits. This finite-depth exponent is not an
asymptotic irrationality bound. The rank-3 recurrence helper is retained as an
independent check, while this notebook uses the full field.


In [ ]:
cmf = pFq(8, 7, 1)
start = dict(zip(x + y, [8] + [3]*7 + [6]*7))
trajectory = dict(zip(x + y, [3] + [1]*7 + [2]*7))
depths = list(range(10, 1001, 10))  # source indices, not walk lengths
seeds = zudilin_zeta5_hypergeometric_seeds()
walks = cmf.walk(trajectory, [N - 2 for N in depths], start)
approximants = [zudilin_zeta5_hypergeometric_readout(W, N, seeds)
                for N, W in zip(depths, walks)]
deltas = [delta(r) for r in approximants]
results["Zudilin (2002)"] = (depths, deltas)
print(f"delta at source index 1000 (998 steps): {deltas[-1]:.12f}")


delta at source index 1000 (998 steps): -0.218905438894


## 2. Brown–Zudilin (2022)

F. Brown and W. Zudilin, *On cellular rational approximations to
$\zeta(5)$*, arXiv:2210.03391 (2022).
[Preprint](https://arxiv.org/abs/2210.03391).

**CMF:** ${}_8F_7(1)$ · **Start:** $(5,2,2,2,2,2,2,2;4,4,4,4,4,4,4)$ ·
**Direction:** $(41,17,16,15,14,13,12,11;24,25,26,27,28,29,30)$.
We use the inverse-transpose walk and the exact two-row projection stored
with the original construction.

In [3]:
cmf = pFq(8, 7, 1)
start = dict(zip(x + y, [5, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4]))
trajectory = dict(zip(x + y, [41, 17, 16, 15, 14, 13, 12, 11, 24, 25, 26, 27, 28, 29, 30]))
depths = list(range(10, 301, 10))
# The dual walk is the inverse transpose of the walk. The symbolic inverse of this
# rank-7 matrix is slow, so the walk is inverted numerically (FLINT).
matrix = cmf.trajectory_matrix(trajectory, start)
walks = NumericMatrix.walk(matrix, Position({n: 1}), depths, Position({n: 0}))
duals = [W.inv().transpose() for W in walks]
approximants = [sum(c * D[j, 0] for j, c in enumerate(NUMERATOR))
                / sum(c * D[j, 0] for j, c in enumerate(DENOMINATOR)) for D in duals]
deltas = [delta(r) for r in approximants]
results["Brown–Zudilin (2022)"] = (depths, deltas)
print(f"delta at depth 300: {deltas[-1]:.9f}")

delta at depth 300: -0.133965704


## 3. New ${}_8F_7(1)$ trajectory

This is the improved $\zeta(5)$ trajectory recorded by the local depth-scan
experiment `improved_m0_131812537914`.

**CMF:** ${}_8F_7(1)$ · **Start:** $(5,2,2,2,2,2,2,2;4,4,4,4,4,4,4)$ ·
**Direction:** $(164,67,63,59,56,52,48,44;97,101,105,108,112,116,120)$.
We use the inverse-transpose walk and the same exact projection as the
Brown–Zudilin construction. Because this trajectory is much more expensive,
this example is sampled through depth 100.

In [4]:
cmf = pFq(8, 7, 1)
start = dict(zip(x + y, [5, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4]))
trajectory = dict(zip(x + y, [164, 67, 63, 59, 56, 52, 48, 44, 97, 101, 105, 108, 112, 116, 120]))
depths = list(range(10, 101, 10))
# The dual walk is the inverse transpose of the walk. The symbolic inverse of this
# rank-7 matrix is slow, so the walk is inverted numerically (FLINT).
matrix = cmf.trajectory_matrix(trajectory, start)
walks = NumericMatrix.walk(matrix, Position({n: 1}), depths, Position({n: 0}))
duals = [W.inv().transpose() for W in walks]
approximants = [sum(c * D[j, 0] for j, c in enumerate(NUMERATOR))
                / sum(c * D[j, 0] for j, c in enumerate(DENOMINATOR)) for D in duals]
deltas = [delta(r) for r in approximants]
results["New ${}_8F_7(1)$ trajectory"] = (depths, deltas)
print(f"delta at depth 100: {deltas[-1]:.9f}")

delta at depth 100: -0.133014805


## All constructions

$\delta_N$ against the walk depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published result | Verified finite-depth delta |
|:--|:--|:--|
| Zudilin (2002) | Error per index tends to a factor below $1.42521964\cdot10^{-4}$; no irrationality-measure bound | $\delta_{1000}=-0.218905438894$ |
| Brown–Zudilin (2022) | Cellular rational approximation construction | Existing computation stops at 300; not checked at 1000 |
| This work | Improved local ${}_8F_7(1)$ trajectory | Existing computation stops at 100; not checked at 1000 |

These constructions are compared as rational approximations. Negative finite-depth exponents are included; they do not establish irrationality. The plot includes all three constructions when their cells are run. Only the new construction was rerun for this recovery; previous construction outputs are preserved.
